# Milestone 2 RAG Implementation

In [12]:
import transformers
from transformers import pipeline

import os
from pathlib import Path
import pandas as pd
import duckdb
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import pickle

In [4]:
if Path.cwd().name != "DSCI_575_project_jt8919_nicolelink33":
    project_root = Path.cwd().parent 
    os.chdir(project_root)
    
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/jennifertsang/Documents/MDS/DSCI 575 Advanced ML/DSCI_575_project_jt8919_nicolelink33


In [5]:
from src.utils import simple_tokenize, display_results
from src.bm25 import search_bm25_with_scores
from src.semantic import search_semantic_with_scores

In [8]:
DATA_DIR = Path("data")
CATEGORY = "Arts_Crafts_and_Sewing"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"
bm25_path = "models/bm25_metadata_index_2.pkl"
semantic_path = "models/faiss_index"
OUTPUT = 'data/processed/stratified_sample.parquet'

## Setup

In [9]:
# 1. Load your newly minted stratified sample
df_sample = pd.read_parquet(OUTPUT)

# 2. Fill any random missing text values to prevent concatenation crashes
text_columns = ['product_title', 'main_category', 'text']
for col in text_columns:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].fillna("")

# 3. Engineer the hybrid page_content string
df_sample['page_content'] = (
    "Product Title: " + df_sample['product_title'] + "\n" +
    "Category: " + df_sample['main_category'] + "\n" +
    "Review Text: " + df_sample['text']
)

# 4. Drop the redundant text columns so they don't clutter the metadata
df_clean = df_sample.drop(columns=['product_title', 'main_category', 'text'])

# 5. Convert the DataFrame into LangChain Documents
documents = []
for _, row in df_clean.iterrows():
    # The engineered string becomes the searchable content
    content = row['page_content']
    
    # Every other column (rating, price, helpful_vote, rating_bucket, len_tier) 
    # gets scooped up into the metadata dictionary!
    metadata = row.drop('page_content').to_dict()
    
    # Create and append the Document
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

print(f"Successfully created {len(documents)} LangChain Documents!")
print("\n--- Let's peek at the first one ---")
print(f"CONTENT:\n{documents[0].page_content}")
print(f"METADATA:\n{documents[0].metadata}")

Successfully created 641 LangChain Documents!

--- Let's peek at the first one ---
CONTENT:
Product Title: Leather Repair Patch Tape Kit Coffee Dark Brown 4 x 60 inch Self Adhesive Leather Repair Patch for Furniture, Couch, Sofa, Car Seats, Office Chair Vinyl Repair Kit
Category: Industrial & Scientific
Review Text: Didn't adhere on lawn mower seat very well. Came loose after one week.
METADATA:
{'rating': 3.0, 'verified_purchase': True, 'helpful_vote': 1, 'parent_asin': 'B09L1JT7SV', 'price': 11.68, 'average_rating': 4.0, 'rating_bucket': '3.7-4.0', 'len_tier': 'short'}


## RAG Explorations

#### With Semantic Retriever

In [ ]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B",
    trust_remote_code=True,
    max_new_tokens=128,
    do_sample=True,
)

llm = HuggingFacePipeline(pipeline=generator)

Fetching 2 files:   0%|          | 0/2 [00:43<?, ?it/s]


### Describe your choice
Document your model choice and rationale in your `results/milestone2_discussion.md` file.

#### With Hybrid Retriever